What is a Trigger?

A Trigger is a stored PL/SQL block that automatically executes when an INSERT, UPDATE, or DELETE event occurs on a table or view.

Why Do We Use Triggers?
Triggers are mainly used for:
1 Auditing
Track who changed what and when.
Example:
Salary changed ↓ Store old salary and new salary in audit table
2 Data Validation
Prevent invalid data.
Example:
Salary < 0 ↓ Reject insert
3 Automatic Logging 
Employee Added 
↓ 
Store activity in log table
4 Business Rules Enforcement
Example:
Employee cannot be deleted after retirement.
Trigger can enforce this rule.

Types of Triggers
1. BEFORE Trigger
Runs before operation.
BEFORE INSERT
BEFORE UPDATE
BEFORE DELETE


Used for:
Validation
Modification
Business Rules

Example:
Check salary
Check age
Check mandatory fields

2. AFTER Trigger
Runs after operation.
AFTER INSERT
AFTER UPDATE
AFTER DELETE

Used for:
Audit
Logging
Tracking
Notifications

3 Row-Level Trigger
FOR EACH ROW
Example:
AFTER INSERT
ON employee
FOR EACH ROW

Suppose:

INSERT 100 rows
Trigger executes:
100 times
One for each row.

4 Statement-Level Trigger
Without:
FOR EACH ROW


Example:
AFTER INSERT ON employee
Insert:
100 rows

Trigger executes:
1 time only
Procedure vs Trigger
This is asked very frequently.

====Trigger Syntax::::

CREATE OR REPLACE TRIGGER trigger_name
BEFORE INSERT
ON employee
FOR EACH ROW
BEGIN
-- logic
END;
/

Question 1: 

our company manages transactions for various financial accounts, to maintain a transaction log and ensure the integrity of the accounts we need to implement a trigger that automatically updates the account balance whenever a new transaction is recorded, each transaction can either be  debit or credit. 
Write a sql server trigger or PL\SQL trigger that updates the accountbalance in the accounts table whenever a new record is inserted into the transcations table 
the trigger should ensure that the balance does not go below zero for any account and raise an exception id this condition is violated-- 
also explain the step by step process of the code and give me good understanding of the logic

**NOTE: This applicable when we have transaction_type in the transcations table**

\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
CREATE OR REPLACE TRIGGER trg_before_transaction
BEFORE INSERT ON transactions
FOR EACH ROW
DECLARE
    v_balance accounts.accountbalance%TYPE;

BEGIN
    -- Get current account balance
    SELECT accountbalance
    INTO v_balance
    FROM accounts
    WHERE account_id = :NEW.account_id
    FOR UPDATE;

    -- Credit Transaction
    IF UPPER(:NEW.transaction_type) = 'CREDIT' THEN

        UPDATE accounts
        SET accountbalance = accountbalance + :NEW.amount
        WHERE account_id = :NEW.account_id;

    -- Debit Transaction
    ELSIF UPPER(:NEW.transaction_type) = 'DEBIT' THEN

        IF (v_balance - :NEW.amount) < 0 THEN

            RAISE_APPLICATION_ERROR(
                -20001,
                'Insufficient Balance. Transaction Rejected.'
            );

        ELSE
            UPDATE accounts
            SET accountbalance = accountbalance - :NEW.amount
            WHERE account_id = :NEW.account_id;

        END IF;

    ELSE
        RAISE_APPLICATION_ERROR(
            -20002,
            'Invalid Transaction Type.'
        );

    END IF;

EXCEPTION

    WHEN NO_DATA_FOUND THEN
        RAISE_APPLICATION_ERROR(
            -20003,
            'Account ID does not exist.'
        );

END trg_before_transaction;
/

----------------------------

FOR UPDATE, added to a cursor's SELECT statement, locks the selected 
rows so no other session can modify them until the transaction is committed or 
rolled back.



Eplination to code============================================================



CREATE OR REPLACE TRIGGER trg_before_transaction
BEFORE INSERT--- (RUNS FIRST INSERTS LATER)
ON transactions--(oracle attaches the Trigger to this transaction table)
FOR EACH ROW(runs each row individually)

DECLARE 
v_balance accounts.balance%TYPE(v_balance is just temporary memory suppose account balance is 5000 oracle stores v_balnce=5000 until trigger finishes and after trigger completes oracle destroys this variable)
%TYPE---  many begineers ask why not write  number -example balance number(10,2) later dba changes it to balance Number(15,4) if Trigger has NUMBER you need to modify code But %type automatically copies datatype
if the column type changes later, the trigger automatically uses the new type.

BEGIN-- everything after begin is executable code
--Get the current balance of the account

SELECT balance  -- read only balance column 
INTO v_balance (example:  read 5000 and store inside v_balance and vbalance=5000
FROM accounts
where account_id=:NEW.account_id;(New is pseudo record available in row level triggers.it contains the values of the row currently being inserted or updated,when a row is being inserted, oracle creates a temporary record called :NEW that contains the values of that row

--if transaction is CREDIT
if condition checks whether a transACTION IS CREDIT OR DEBIT
IF UPPER(:NEW.transaction_type) = 'CREDIT' THEN
Upper() converts the text to uppercase
UPDATE accounts
SET balance = balance + :NEW.amount
where account_id = :NEW.account_id;


----if transaction is credit
IF UPPER(:NEW.transaction_type) = 'CREDIT' THEN

UPDATE accounts
SET balance = balance + :NEW.amount
where account_id= :NEW.account_id;

---if transaction is Debit

ELSIF UPPER(:NEW.transaction_type) = 'DEBIT' THEN

--check if sufficient balance exists

IF v_balance >= :NEW.amount THEN

UPDATE accounts
SET balance = balance - :NEW.amount
where account_id = :NEW.account_id;

ELSE

RAISE_APPLICATION_ERROR
(
-2001,'Insufficient Balance. Transaction Rejected.'
);

END IF;-- FIRST END IF Closes the inner balance check condition;

ELSE

RAISE_APPLICATION_ERROR
(
-2002,'Invalid Transaction Type.'
);

END IF;- the second end if closes the outer transaction type condition

END;-- marks the end of the trigger body


Question 2 book and inventory
table 1 books: book_id, title, author, inventory
table 2 sales: sales_id, book_id, quantity_sold, sale_date



CREATE OR REPLACE TRIGGER update_inventory
BEFORE INSERT ON sales
FOR EACH ROW

DECLARE
    v_stock books.inventory%TYPE;

BEGIN
    -- Get current inventory for the book
    SELECT inventory
    INTO v_stock
    FROM books
    WHERE book_id = :NEW.book_id;

    -- Check whether inventory falls below 10 after sale
    IF (v_stock - :NEW.quantity_sold) < 10 THEN
        RAISE_APPLICATION_ERROR(
            -20003,
            'Stock upgrade required. Inventory cannot be less than 10.'
        );

    ELSE
        -- Reduce inventory
        UPDATE books
        SET inventory = inventory - :NEW.quantity_sold
        WHERE book_id = :NEW.book_id;

    END IF;

EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RAISE_APPLICATION_ERROR(
            -20004,
            'No Book ID Found.'
        );

END update_inventory;
/

Q3) Account Balance Trigger: 
Create a trigger to update the account balance whenever a new transaction is inserted. 
There are two tables: an Accounts table containing the account balance and a Transactions 
table containing the transaction amount. The transaction amount can be either "positive or 
negative". The trigger should check the current account balance against the transaction 
amount. If the transaction would result in an insufficient balance, raise an application error; 
otherwise, update the account balance accordingly.


ORRRRRRRRRRRRRRRRRRRRRRRRR

Our company manages transactions for various financial accounts. To maintain a
transaction log and ensure the integrity of the accounts, we need to implement a
trigger that automatically updates the account balance whenever a new transaction
is recorded. Each transaction can either be a debit or a credit.

Write a PL/SQL trigger that updates the `account_balance` in the `accounts` table
whenever a new record is inserted into the `transactions` table. The trigger
should ensure that the balance does not go below zero for any account and raise an
exception if this condition is violated.

-- Creating tables

CREATE TABLE accounts (
    account_id INT PRIMARY KEY,
    account_name VARCHAR(100),
    account_balance DECIMAL(10, 2)
);

CREATE TABLE transactions (
    transaction_id INT PRIMARY KEY,
    account_id INT,
    transaction_amount DECIMAL(10, 2),
    transaction_date DATE,
    CONSTRAINT fk_account
        FOREIGN KEY (account_id)
        REFERENCES accounts (account_id)
);



CREATE OR REPLACE TRIGGER trg_update_balance
BEFORE INSERT ON transactions
FOR EACH ROW
DECLARE
    v_balance accounts.account_balance%TYPE;

BEGIN
    -- Get current account balance
    SELECT account_balance
    INTO v_balance
    FROM accounts
    WHERE account_id = :NEW.account_id
    FOR UPDATE;

    -- Check whether balance becomes negative
    IF (v_balance + :NEW.transaction_amount) < 0 THEN

        RAISE_APPLICATION_ERROR(
            -20001,
            'Insufficient Balance'
        );

    ELSE

        -- Update account balance
        UPDATE accounts
        SET account_balance =
            account_balance + :NEW.transaction_amount
        WHERE account_id = :NEW.account_id;
        

    END IF;

EXCEPTION

    WHEN NO_DATA_FOUND THEN
        RAISE_APPLICATION_ERROR(
            -20002,
            'Account ID does not exist'
        );

END trg_update_balance;
/